In [36]:
import mysql.connector
mydb = mysql.connector.connect(host="localhost",user="root",password="",)
print(mydb)
mycursor = mydb.cursor(buffered=True)

In [ ]:
mycursor.execute("CREATE DATABASE Sales1")

DatabaseError: 1007 (HY000): Can't create database 'sales1'; database exists

In [ ]:
mycursor.execute("use sales1")

In [ ]:
mycursor.execute("""create table branches
                 ( branch_id int auto_increment primary key,
                 branch_name VARCHAR(100),
                 branch_admin_name VARCHAR(100))""")

In [ ]:
mycursor.execute("""create table customer_sales
                 (sale_id int auto_increment primary key,
                 branch_id int,
                 date DATE,
                 name VARCHAR(100),
                 mobile_number VARCHAR(100) UNIQUE,
                 product_name VARCHAR(100),
                 gross_sales DECIMAL(12,2),
                 received_amount DECIMAL(12,2) DEFAULT 0,
                 pending_amount DECIMAL(12,2)
                    GENERATED ALWAYS AS (gross_sales - received_amount) STORED,
                 status ENUM('Open','Close'),
                 FOREIGN KEY (branch_id) REFERENCES branches(branch_id))""")

In [ ]:
mycursor.execute(""" create table users
                 (user_id INT auto_increment primary key,
                 username varchar(100),
                 password varchar(300),
                 branch_id int NULL,
                 role ENUM('Super Admin', 'Admin'),
                 email VARCHAR(255) UNIQUE,
                 FOREIGN KEY (branch_id) REFERENCES branches(branch_id))""")

In [ ]:
mycursor.execute("""create table payment_splits
                 (payment_id int auto_increment primary key,
                 sale_id int,
                 payment_date DATE,
                 amount_paid DECIMAL(12,2),
                 payment_method VARCHAR(50),
                 FOREIGN KEY (sale_id) REFERENCES customer_sales(sale_id))""")

In [ ]:
mycursor.execute("""
CREATE TRIGGER update_received_amount
AFTER INSERT ON payment_splits
FOR EACH ROW
UPDATE customer_sales
SET received_amount = (
    SELECT IFNULL(SUM(amount_paid), 0)
    FROM payment_splits
    WHERE sale_id = NEW.sale_id
)
WHERE sale_id = NEW.sale_id
""")

In [ ]:
mycursor.execute("SELECT * FROM customer_sales")
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-----------+-------------+------------+---------------+-----------------+----------------+---------------+-------------------+------------------+----------+
|   sale_id |   branch_id | date       | name          |   mobile_number | product_name   |   gross_sales |   received_amount |   pending_amount | status   |
|-----------+-------------+------------+---------------+-----------------+----------------+---------------+-------------------+------------------+----------|
|         1 |           2 | 2024-01-02 | Customer_1    |      9800000001 | DS             |         40000 |              7296 |            32704 | Open     |
|         2 |           5 | 2024-01-03 | Customer_2    |      9800000002 | BA             |         30000 |              7314 |            22686 | Open     |
|         3 |           4 | 2024-01-04 | Customer_3    |      9800000003 | DA             |         35000 |              5697 |            29303 | Open     |
|         4 |           2 | 2024-01-05 | Customer_4 

In [ ]:
mycursor.execute("SELECT * FROM branches")
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-------------+---------------+---------------------+
|   branch_id | branch_name   | branch_admin_name   |
|-------------+---------------+---------------------|
|           1 | Chennai       | Arun Kumar          |
|           2 | Bangalore     | Ravi Shankar        |
|           3 | Hyderabad     | Suresh Reddy        |
|           4 | Delhi         | Neha Sharma         |
|           5 | Mumbai        | Rahul Mehta         |
|           6 | Pune          | Amit Patil          |
|           7 | Kolkata       | Subham Ghosh        |
|           8 | Ahmedabad     | Raj Patel           |
+-------------+---------------+---------------------+


In [ ]:
mycursor.execute("SELECT * FROM payment_splits")
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+--------------+-----------+----------------+---------------+------------------+
|   payment_id |   sale_id | payment_date   |   amount_paid | payment_method   |
|--------------+-----------+----------------+---------------+------------------|
|            1 |         1 | 2024-01-06     |          7296 | Cash             |
|            2 |         2 | 2024-01-04     |          7314 | Card             |
|            3 |         3 | 2024-01-04     |          3457 | Cash             |
|            4 |         3 | 2024-01-07     |           384 | Cash             |
|            5 |         3 | 2024-01-12     |          1856 | Card             |
|            6 |         4 | 2024-01-15     |           408 | Card             |
|            7 |         4 | 2024-01-11     |          1117 | Cash             |
|            8 |         4 | 2024-01-12     |           214 | Card             |
|            9 |         5 | 2024-01-08     |         18231 | Card             |
|           10 |         6 |

In [ ]:
mycursor.execute("SELECT * FROM users")
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-----------+-----------------+------------+-------------+-------------+------------------------+
|   user_id | username        | password   |   branch_id | role        | email                  |
|-----------+-----------------+------------+-------------+-------------+------------------------|
|         1 | superadmin      | super123   |             | Super Admin | superadmin@company.com |
|         2 | admin_chennai   | admin123   |           1 | Admin       | chennai@company.com    |
|         3 | admin_bangalore | admin123   |           2 | Admin       | bangalore@company.com  |
|         4 | admin_hyderabad | admin123   |           3 | Admin       | hyderabad@company.com  |
|         5 | admin_delhi     | admin123   |           4 | Admin       | delhi@company.com      |
|         6 | admin_mumbai    | admin123   |           5 | Admin       | mumbai@company.com     |
|         7 | admin_pune      | admin123   |           6 | Admin       | pune@company.com       |
|         8 | admin_

In [ ]:
#SQL QUESTIONS
#Basic Queries - Answer any 4 questions
#Display all sales with status = 'Open'
mycursor.execute("SELECT * FROM customer_sales WHERE status = 'Open'")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+-----------+-------------+------------+---------------+-----------------+----------------+---------------+-------------------+------------------+----------+
|   sale_id |   branch_id | date       | name          |   mobile_number | product_name   |   gross_sales |   received_amount |   pending_amount | status   |
|-----------+-------------+------------+---------------+-----------------+----------------+---------------+-------------------+------------------+----------|
|         1 |           2 | 2024-01-02 | Customer_1    |      9800000001 | DS             |         40000 |              7296 |            32704 | Open     |
|         2 |           5 | 2024-01-03 | Customer_2    |      9800000002 | BA             |         30000 |              7314 |            22686 | Open     |
|         3 |           4 | 2024-01-04 | Customer_3    |      9800000003 | DA             |         35000 |              5697 |            29303 | Open     |
|         4 |           2 | 2024-01-05 | Customer_4 

In [ ]:
#Retrieve all sales belonging to the Chennai branch
mycursor.execute("""
SELECT * FROM customer_sales
WHERE branch_id = (SELECT branch_id FROM branches WHERE branch_name = 'Chennai')""")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+-----------+-------------+------------+--------------+-----------------+----------------+---------------+-------------------+------------------+----------+
|   sale_id |   branch_id | date       | name         |   mobile_number | product_name   |   gross_sales |   received_amount |   pending_amount | status   |
|-----------+-------------+------------+--------------+-----------------+----------------+---------------+-------------------+------------------+----------|
|         6 |           1 | 2024-01-07 | Customer_6   |      9800000006 | FSD            |         45000 |             27696 |            17304 | Open     |
|         8 |           1 | 2024-01-09 | Customer_8   |      9800000008 | BA             |         30000 |             26447 |             3553 | Open     |
|        11 |           1 | 2024-01-12 | Customer_11  |      9800000011 | DA             |         35000 |             24911 |            10089 | Open     |
|        23 |           1 | 2024-01-24 | Customer_23  |   

In [ ]:
#Calculate the total gross sales across all branches.
mycursor.execute("SELECT SUM(gross_sales) as gross_sales FROM customer_sales")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))


+---------------+
|   gross_sales |
|---------------|
|    3.6768e+07 |
+---------------+


In [ ]:
#Calculate the total received amount across all sales.
mycursor.execute("SELECT SUM(received_amount) as received_amount FROM customer_sales")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+-------------------+
|   received_amount |
|-------------------|
|       1.90281e+07 |
+-------------------+


In [ ]:
#Calculate the total pending amount across all sales.
mycursor.execute("SELECT SUM(pending_amount) as pending_amount FROM customer_sales")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+------------------+
|   pending_amount |
|------------------|
|      1.77399e+07 |
+------------------+


In [ ]:
#Count the total number of sales per branch.
mycursor.execute("""
SELECT branch_id, COUNT(*) 
FROM customer_sales
GROUP BY branch_id
""")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+-------------+------------+
|   branch_id |   COUNT(*) |
|-------------+------------|
|           1 |        110 |
|           2 |        133 |
|           3 |        106 |
|           4 |        134 |
|           5 |        136 |
|           6 |        120 |
|           7 |        133 |
|           8 |        128 |
+-------------+------------+


In [ ]:
#Retrieve sales details along with the branch name.
mycursor.execute("""
SELECT customer_sales.sale_id, branches.branch_name, customer_sales.gross_sales
FROM customer_sales
JOIN branches ON customer_sales.branch_id = branches.branch_id
""")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+-----------+---------------+---------------+
|   sale_id | branch_name   |   gross_sales |
|-----------+---------------+---------------|
|         6 | Chennai       |         45000 |
|         8 | Chennai       |         30000 |
|        11 | Chennai       |         35000 |
|        23 | Chennai       |         30000 |
|        35 | Chennai       |         30000 |
|        36 | Chennai       |         48000 |
|        57 | Chennai       |         42000 |
|        63 | Chennai       |         45000 |
|        79 | Chennai       |         45000 |
|        80 | Chennai       |         30000 |
|       101 | Chennai       |         40000 |
|       106 | Chennai       |         35000 |
|       113 | Chennai       |         40000 |
|       115 | Chennai       |         28000 |
|       121 | Chennai       |         45000 |
|       136 | Chennai       |         42000 |
|       144 | Chennai       |         45000 |
|       146 | Chennai       |         40000 |
|       150 | Chennai       |     

In [ ]:
#Retrieve sales details along with total payment received (using payment_splits)
mycursor.execute("""
SELECT c.sale_id, SUM(p.amount_paid) as amount_paid
FROM customer_sales c
JOIN payment_splits p ON c.sale_id = p.sale_id
GROUP BY c.sale_id
""")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+-----------+---------------+
|   sale_id |   amount_paid |
|-----------+---------------|
|         1 |          7296 |
|         2 |          7314 |
|         3 |          5697 |
|         4 |          1739 |
|         5 |         18231 |
|         6 |         27696 |
|         7 |          3039 |
|         8 |         26447 |
|         9 |          4090 |
|        10 |         23700 |
|        11 |         24911 |
|        12 |         23283 |
|        13 |          4679 |
|        14 |         41943 |
|        15 |         26290 |
|        16 |         42954 |
|        17 |          9150 |
|        18 |         17219 |
|        19 |          5957 |
|        20 |         41120 |
|        21 |         39086 |
|        22 |         39052 |
|        23 |         23616 |
|        24 |         17486 |
|        25 |         17261 |
|        26 |         13035 |
|        27 |         35348 |
|        28 |         15692 |
|        29 |          2580 |
|        30 |         34911 |
|        3

In [ ]:
#Show branch-wise total gross sales (using JOIN & GROUP BY).

mycursor.execute("""
SELECT b.branch_name, SUM(c.gross_sales) as Gross_sales
FROM customer_sales c
JOIN branches b ON c.branch_id = b.branch_id
GROUP BY b.branch_name
""")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+---------------+---------------+
| branch_name   |   Gross_sales |
|---------------+---------------|
| Ahmedabad     |     4.695e+06 |
| Bangalore     |     4.71e+06  |
| Chennai       |     4.128e+06 |
| Delhi         |     4.914e+06 |
| Hyderabad     |     3.809e+06 |
| Kolkata       |     5.101e+06 |
| Mumbai        |     4.853e+06 |
| Pune          |     4.558e+06 |
+---------------+---------------+


In [ ]:
#Retrieve sales along with branch admin name.
mycursor.execute("""
SELECT c.sale_id, b.branch_admin_name
FROM customer_sales c
JOIN branches b ON c.branch_id = b.branch_id
""")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+-----------+---------------------+
|   sale_id | branch_admin_name   |
|-----------+---------------------|
|         6 | Arun Kumar          |
|         8 | Arun Kumar          |
|        11 | Arun Kumar          |
|        23 | Arun Kumar          |
|        35 | Arun Kumar          |
|        36 | Arun Kumar          |
|        57 | Arun Kumar          |
|        63 | Arun Kumar          |
|        79 | Arun Kumar          |
|        80 | Arun Kumar          |
|       101 | Arun Kumar          |
|       106 | Arun Kumar          |
|       113 | Arun Kumar          |
|       115 | Arun Kumar          |
|       121 | Arun Kumar          |
|       136 | Arun Kumar          |
|       144 | Arun Kumar          |
|       146 | Arun Kumar          |
|       150 | Arun Kumar          |
|       152 | Arun Kumar          |
|       156 | Arun Kumar          |
|       162 | Arun Kumar          |
|       171 | Arun Kumar          |
|       176 | Arun Kumar          |
|       195 | Arun Kumar    

In [ ]:
#Find sales where the pending amount is greater than 5000
mycursor.execute("select * from customer_sales where pending_amount > 5000 limit 10")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+-----------+-------------+------------+-------------+-----------------+----------------+---------------+-------------------+------------------+----------+
|   sale_id |   branch_id | date       | name        |   mobile_number | product_name   |   gross_sales |   received_amount |   pending_amount | status   |
|-----------+-------------+------------+-------------+-----------------+----------------+---------------+-------------------+------------------+----------|
|         1 |           2 | 2024-01-02 | Customer_1  |      9800000001 | DS             |         40000 |              7296 |            32704 | Open     |
|         2 |           5 | 2024-01-03 | Customer_2  |      9800000002 | BA             |         30000 |              7314 |            22686 | Open     |
|         3 |           4 | 2024-01-04 | Customer_3  |      9800000003 | DA             |         35000 |              5697 |            29303 | Open     |
|         4 |           2 | 2024-01-05 | Customer_4  |      9800

In [ ]:
#Retrieve top 3 highest gross sales.
mycursor.execute("select * from customer_sales ORDER BY gross_sales desc limit 3")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+-----------+-------------+------------+--------------+-----------------+----------------+---------------+-------------------+------------------+----------+
|   sale_id |   branch_id | date       | name         |   mobile_number | product_name   |   gross_sales |   received_amount |   pending_amount | status   |
|-----------+-------------+------------+--------------+-----------------+----------------+---------------+-------------------+------------------+----------|
|       771 |           6 | 2024-02-11 | Customer_771 |      9800000771 | AI             |         48000 |              8937 |            39063 | Open     |
|       774 |           3 | 2024-02-14 | Customer_774 |      9800000774 | AI             |         48000 |             33590 |            14410 | Open     |
|       519 |           8 | 2024-06-03 | Customer_519 |      9800000519 | AI             |         48000 |              6237 |            41763 | Open     |
+-----------+-------------+------------+--------------+---

In [ ]:
#Calculate payment method-wise total collection (Cash / UPI / Card).
mycursor.execute("SELECT payment_method, SUM(amount_paid) as amount_paid FROM payment_splits GROUP BY payment_method")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+------------------+---------------+
| payment_method   |   amount_paid |
|------------------+---------------|
| Card             |   6.60801e+06 |
| Cash             |   6.38701e+06 |
| UPI              |   6.0331e+06  |
+------------------+---------------+


In [ ]:
#Retrieve monthly sales summary (group by month & year).
mycursor.execute("SELECT YEAR(date), MONTH(date), SUM(gross_sales) as Gross_sales FROM customer_sales GROUP BY YEAR(date), MONTH(date)")
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,headers=[i[0] for i in mycursor.description],  tablefmt='psql'))

+--------------+---------------+---------------+
|   YEAR(date) |   MONTH(date) |   Gross_sales |
|--------------+---------------+---------------|
|         2024 |             1 |     3.487e+06 |
|         2024 |             2 |     3.116e+06 |
|         2024 |             3 |     3.442e+06 |
|         2024 |             4 |     3.272e+06 |
|         2024 |             5 |     3.418e+06 |
|         2024 |             6 |     3.294e+06 |
|         2024 |             7 |     3.517e+06 |
|         2024 |             8 |     3.254e+06 |
|         2024 |             9 |     3.183e+06 |
|         2024 |            10 |     2.362e+06 |
|         2024 |            11 |     2.248e+06 |
|         2024 |            12 |     2.175e+06 |
+--------------+---------------+---------------+
